In [36]:
import sys
from pathlib import Path
import pandas as pd
import json

projectRoot = Path().resolve().parents[0]
print(f"path: {projectRoot}")
sys.path.append(str(projectRoot))

%load_ext autoreload
%autoreload 2

path: /workspace
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [39]:
from src.parsing.arelle_parser import load_model_xbrl
from src.parsing.arelle_parser import (safe_dict, extract_context_rows,
                                       extract_fact_rows)

html_folder = projectRoot / "data" / "raw" / "fca"

# Loop through all HTML files in subfolders within the 'fca' folder   
merged_dfs = []
debug = False
for html_filepath in html_folder.rglob("*.html"):
    if debug and html_filepath.name != "PY6ZZQWO2IZFZC3IOL08-2022-12-31-T01.html":
        continue
    print(f"Processing file: {html_filepath}")
    filing_basefolder = html_filepath.parents[1]
    
    model_xbrl = load_model_xbrl(str(filing_basefolder),
                                 str(html_filepath))

    # Perform operations with model_xbrl (if needed)
    filing_id = html_filepath.parents[1].stem 
    fact_rows = extract_fact_rows(model_xbrl, filing_id)
    context_rows = extract_context_rows(model_xbrl, filing_id)

    # Convert fact_rows and context_rows to pandas DataFrames
    fact_df = pd.DataFrame(fact_rows)
    context_df = pd.DataFrame(context_rows)
    merged_df = pd.merge(fact_df, context_df,
                         on=['filing_id', 'context_id'], how='inner')

    # Append the current merged_df to the list
    merged_dfs.append(merged_df)

# Concatenate all merged DataFrames into a single DataFrame
final_df = pd.concat(merged_dfs, ignore_index=True)

# Save the merged DataFrame as a CSV file
output_path = (projectRoot / "data" / "processed" / 
               "canonical_facts" / "GSK_AZ_3yr_Report.csv")
final_df.to_csv(output_path, index=False)
print(f"Saved merged DataFrame to {output_path}")

Processing file: /workspace/data/raw/fca/Astrazeneca_PLC/AstraZeneca_PLC_2022/reports/PY6ZZQWO2IZFZC3IOL08-2022-12-31-T01.html
Processing file: /workspace/data/raw/fca/Astrazeneca_PLC/AstraZeneca_PLC_2023/reports/PY6ZZQWO2IZFZC3IOL08-2023-12-31-T01.html
Processing file: /workspace/data/raw/fca/Astrazeneca_PLC/AstraZeneca_PLC_2024/reports/PY6ZZQWO2IZFZC3IOL08-2024-12-31-T01.html
Processing file: /workspace/data/raw/fca/GSK_PLC/GSK_PLC-2022/reports/5493000HZTVUYLO1D793-2022-12-31-T01.html
Processing file: /workspace/data/raw/fca/GSK_PLC/GSK_PLC-2023/reports/5493000HZTVUYLO1D793-2023-12-31-T01.html
Processing file: /workspace/data/raw/fca/GSK_PLC/GSK_PLC-2024/reports/5493000HZTVUYLO1D793-2024-12-31-T01.html
Saved merged DataFrame to /workspace/data/processed/canonical_facts/GSK_AZ_3yr_Report.csv


In [38]:
print(final_df.keys())
print(final_df["filing_id"].value_counts())
print(final_df["unit_ref"].value_counts())
print(final_df["taxonomy_domain"].value_counts())

Index(['filing_id', 'context_id', 'raw_name', 'taxonomy_domain', 'data_type',
       'unit_ref', 'decimals', 'value_text', 'value_numeric', 'entity_scheme',
       'entity_identifier', 'period_type', 'instant', 'period_start',
       'period_end', 'dimensional_qualifier'],
      dtype='object')
filing_id
GSK_PLC-2024            933
GSK_PLC-2022            860
GSK_PLC-2023            857
AstraZeneca_PLC_2024    856
AstraZeneca_PLC_2023    839
AstraZeneca_PLC_2022    829
Name: count, dtype: int64
unit_ref
Unit_USD              1984
GBP                   1469
Unit_GBP               717
GBP_per_Share           57
Unit_USD_per_Share      42
Unit_shares             18
Name: count, dtype: int64
taxonomy_domain
ifrs-full    4256
gsk           503
azn           415
Name: count, dtype: int64


In [14]:
final_df[final_df["raw_name"]=="ifrs-full:BasicEarningsLossPerShare"]

,filing_id,context_id,raw_name,taxonomy_domain,data_type,unit_ref,decimals,value_text,value_numeric,entity_scheme,entity_identifier,period_type,instant,period_start,period_end,dimensional_qualifier
90,GSK_PLC-2022,P01_01_2022To12_31_2022,ifrs-full:BasicEarningsLossPerShare,ifrs-full,string,GBP_per_Share,3,3.714,None,http://standards.iso.org/iso/17442,5493000HZTVUYLO1D793,duration,None,2022-01-01,2023-01-01,{}
91,GSK_PLC-2022,P01_01_2021To12_31_2021,ifrs-full:BasicEarningsLossPerShare,ifrs-full,string,GBP_per_Share,3,1.096,None,http://standards.iso.org/iso/17442,5493000HZTVUYLO1D793,duration,None,2021-01-01,2022-01-01,{}
92,GSK_PLC-2022,P01_01_2020To12_31_2020,ifrs-full:BasicEarningsLossPerShare,ifrs-full,string,GBP_per_Share,3,1.444,None,http://standards.iso.org/iso/17442,5493000HZTVUYLO1D793,duration,None,2020-01-01,2021-01-01,{}
948,GSK_PLC-2023,c-1,ifrs-full:BasicEarningsLossPerShare,ifrs-full,string,GBP_per_Share,3,1.216,None,http://standards.iso.org/iso/17442,5493000HZTVUYLO1D793,duration,None,2023-01-01,2024-01-01,{}
949,GSK_PLC-2023,c-2,ifrs-full:BasicEarningsLossPerShare,ifrs-full,string,GBP_per_Share,3,3.714,None,http://standards.iso.org/iso/17442,5493000HZTVUYLO1D793,duration,None,2022-01-01,2023-01-01,{}
950,GSK_PLC-2023,c-3,ifrs-full:BasicEarningsLossPerShare,ifrs-full,string,GBP_per_Share,3,1.096,None,http://standards.iso.org/iso/17442,5493000HZTVUYLO1D793,duration,None,2021-01-01,2022-01-01,{}
1804,GSK_PLC-2024,c-1,ifrs-full:BasicEarningsLossPerShare,ifrs-full,string,GBP_per_Share,3,0.632,None,http://standards.iso.org/iso/17442,5493000HZTVUYLO1D793,duration,None,2024-01-01,2025-01-01,{}
1805,GSK_PLC-2024,c-2,ifrs-full:BasicEarningsLossPerShare,ifrs-full,string,GBP_per_Share,3,1.216,None,http://standards.iso.org/iso/17442,5493000HZTVUYLO1D793,duration,None,2023-01-01,2024-01-01,{}
1806,GSK_PLC-2024,c-3,ifrs-full:BasicEarningsLossPerShare,ifrs-full,string,GBP_per_Share,3,3.714,None,http://standards.iso.org/iso/17442,5493000HZTVUYLO1D793,duration,None,2022-01-01,2023-01-01,{}
